# 📈 Time Series Forecasting with Prophet

In this notebook, we will demonstrate how to use **Prophet** (developed by Meta) to forecast sensor data. 

### Why Prophet?
- **Ease of use**: Handles seasonality, holidays, and outliers automatically.
- **Flexibility**: Works well with missing data and trend shifts.
- **Speed**: Fast enough for interactive analysis.

---

## 1. Import Libraries
We'll need `pandas` for data handling, `plotly` for interactive visualizations, and `prophet` for the forecasting model.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from prophet import Prophet
from prophet.plot import plot_plotly, plot_components_plotly
import warnings
warnings.filterwarnings('ignore')

## 2. Load and Prepare Data
We are using the sensor data from August 2023. Since the raw data has over 1 million records, we will **resample** it to hourly averages to make our analysis more manageable and meaningful.

In [3]:
# Load the dataset
file_path = '../data/s1_august_23.csv'
df = pd.read_csv(file_path)

# Convert timestamp to datetime
df['ts'] = pd.to_datetime(df['ts'])

# Resample to hourly average to handle the high-frequency noise
df_hourly = df.set_index('ts')['VALUE'].resample('h').mean().reset_index()

df_hourly.head()

,ts,VALUE
0,2023-08-01 00:00:00,4.022111
1,2023-08-01 01:00:00,4.020614
2,2023-08-01 02:00:00,4.025302
3,2023-08-01 03:00:00,4.030110
4,2023-08-01 04:00:00,4.026218


## 3. Visualize Existing Data
Let's start by plotting the sensor data to understand the trends and patterns.

In [4]:
fig = px.line(df_hourly, x='ts', y='VALUE', 
              title='Sensor Readings - August 2023 (Hourly Average)',
              labels={'ts': 'Time', 'VALUE': 'Sensor Value'},
              template='plotly_dark')

fig.update_traces(line_color='#00d1ff')
fig.show()

: 

## 4. Prepare for Prophet
Prophet requires the input dataframe to have exactly two columns:
- `ds`: The timestamp column.
- `y`: The value we want to forecast.

In [5]:
prophet_df = df_hourly.rename(columns={'ts': 'ds', 'VALUE': 'y'})
prophet_df.head()

,ds,y
0,2023-08-01 00:00:00,4.022111
1,2023-08-01 01:00:00,4.020614
2,2023-08-01 02:00:00,4.025302
3,2023-08-01 03:00:00,4.030110
4,2023-08-01 04:00:00,4.026218


## 5. Train the Model
We initialize the Prophet model and fit it to our data.

In [6]:
m = Prophet(yearly_seasonality=False, weekly_seasonality=True, daily_seasonality=True)
m.fit(prophet_df)

16:26:09 - cmdstanpy - INFO - Chain [1] start processing
16:26:09 - cmdstanpy - INFO - Chain [1] done processing


## 6. Create Forecast
We'll predict the sensor values for the next 7 days (168 hours).

In [8]:
# Create a dataframe for the next 7 days
future = m.make_future_dataframe(periods=168, freq='h')

# Predict
forecast = m.predict(future)

# Show the last few rows of the forecast
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail()

,ds,yhat,yhat_lower,yhat_upper
907,2023-09-07 19:00:00,4.029415,3.925378,4.134290
908,2023-09-07 20:00:00,4.029475,3.925679,4.133864
909,2023-09-07 21:00:00,4.029679,3.923500,4.137241
910,2023-09-07 22:00:00,4.030175,3.922034,4.136292
911,2023-09-07 23:00:00,4.030759,3.922724,4.142613


## 7. Visualize Results
Prophet provides built-in Plotly functions to visualize the forecast and its components.

In [11]:
# Plot the forecast
fig_forecast = plot_plotly(m, forecast)
fig_forecast.update_layout(title='Sensor Value Forecast')
fig_forecast.show()

### Component Analysis
Prophet breaks down the forecast into Trend, Weekly, and Daily components. This is extremely useful for understanding **when** the sensor values typically peak.

In [ ]:
fig_components = plot_components_plotly(m, forecast)
fig_components.update_layout()
fig_components.show()

## 8. Measure Forecast Error with Rolling and Expanding Windows
We can compare Prophet's in-sample predictions to the actual sensor values and compute error metrics over time. Rolling windows highlight recent performance, while expanding windows show how the model improves as more data is included.

In [12]:
# Merge Prophet predictions with the known historical values
forecast_history = forecast[['ds', 'yhat']].merge(prophet_df, on='ds', how='inner')

# Compute common error metrics
forecast_history['error'] = forecast_history['y'] - forecast_history['yhat']
forecast_history['abs_error'] = forecast_history['error'].abs()
forecast_history['squared_error'] = forecast_history['error'] ** 2
forecast_history['mape'] = (forecast_history['abs_error'] / forecast_history['y'].abs()) * 100

# Rolling and expanding metrics
window_hours = 24
forecast_history = forecast_history.set_index('ds').sort_index()
forecast_history['rolling_mae'] = forecast_history['abs_error'].rolling(window=window_hours, min_periods=1).mean()
forecast_history['rolling_rmse'] = (forecast_history['squared_error'].rolling(window=window_hours, min_periods=1).mean() ** 0.5)
forecast_history['expanding_mae'] = forecast_history['abs_error'].expanding(min_periods=1).mean()
forecast_history['expanding_rmse'] = (forecast_history['squared_error'].expanding(min_periods=1).mean() ** 0.5)

forecast_history[['y', 'yhat', 'error', 'abs_error', 'rolling_mae', 'expanding_mae']].head()

,y,yhat,error,abs_error,rolling_mae,expanding_mae
ds,,,,,,
2023-08-01 00:00:00,4.022111,4.022473,-0.000362,0.000362,0.000362,0.000362
2023-08-01 01:00:00,4.020614,4.022814,-0.002200,0.002200,0.001281,0.001281
2023-08-01 02:00:00,4.025302,4.022889,0.002412,0.002412,0.001658,0.001658
2023-08-01 03:00:00,4.030110,4.023115,0.006995,0.006995,0.002992,0.002992
2023-08-01 04:00:00,4.026218,4.023761,0.002457,0.002457,0.002885,0.002885


In [13]:
fig_error = go.Figure()
fig_error.add_trace(go.Scatter(
    x=forecast_history.index,
    y=forecast_history['abs_error'],
    mode='lines',
    name='Absolute Error',
    line=dict(color='#ff7f0e'),
))
fig_error.add_trace(go.Scatter(
    x=forecast_history.index,
    y=forecast_history['rolling_mae'],
    mode='lines',
    name=f'{window_hours}h Rolling MAE',
    line=dict(color='#1f77b4'),
))
fig_error.add_trace(go.Scatter(
    x=forecast_history.index,
    y=forecast_history['expanding_mae'],
    mode='lines',
    name='Expanding MAE',
    line=dict(color='#2ca02c'),
))
fig_error.update_layout(
    title='Rolling vs Expanding Forecast Error',
    xaxis_title='Time',
    yaxis_title='Error',
    template='plotly_dark',
)
fig_error.show()

### What this shows
- `rolling_mae` captures how recent forecast performance changes over the past 24 hours.
- `expanding_mae` shows the error aggregated over the model's entire history up to each point.
- Comparing them helps identify short-term drift versus long-term model stability.